# WTI Hourly Price Data — Yahoo Finance

This notebook downloads two years of hourly OHLCV (Open, High, Low, Close, Volume) 
data for WTI Crude Oil futures (ticker: CL=F) from Yahoo Finance using the yfinance 
library. It then computes four derived liquidity metrics — log volume, price range, 
log return, and the Amihud illiquidity ratio — which serve as the primary dependent 
variables in the thesis analysis. The resulting dataset is saved to 
`01_data/raw/price/wti_hourly_raw.csv`.

### General imports

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3

### Connect to the db

In [11]:
conn = sqlite3.connect("../01_data/wti_thesis.db")
print("Connected to wti_thesis.db")

Connected to wti_thesis.db


### Obtain DXY, VIX information

In [12]:
tickers = {
    "CL=F": "wti",
    "DX-Y.NYB": "dxy",
    "^VIX": "vix"
}

data = {}
for ticker, name in tickers.items():
    df = yf.download(ticker, period="2y", interval="1h", progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index).tz_convert('UTC')
    df.index.name = 'datetime_hour'
    data[name] = df
    print(f"{name.upper()}: {len(df)} records | {df.index.min().date()} to {df.index.max().date()}")

WTI: 11193 records | 2024-05-10 to 2026-05-08
DXY: 11907 records | 2024-05-10 to 2026-05-08
VIX: 7121 records | 2024-05-09 to 2026-05-08


### Obtain liquidity information

- **Open:** Price at th emoment of opening the time window.

- **High:** Highest price in the time window.

- **Low:** Lowest price in the time window.

- **Volume:** Amount of contracts on the time window.

- **Close:** Price at the moment of closing the time window.

In [13]:
df_price = yf.download("CL=F", period="2y", interval="1h", progress=False)

# If Yahoo Finance changes the order of the columns in the future, this avoid silent erros of calculating values with different columns
if isinstance(df_price.columns, pd.MultiIndex):
    df_price.columns = df_price.columns.get_level_values(0)
df_price.index = pd.to_datetime(df_price.index).tz_convert('UTC')
df_price.index.name = 'Datetime'

print(f"Registros: {len(df_price)}")
print(f"Rango: {df_price.index.min()} a {df_price.index.max()}")
print(df_price.tail(5))

Registros: 11193
Rango: 2024-05-10 13:00:00+00:00 a 2026-05-08 20:00:00+00:00
Price                          Close       High        Low       Open  Volume
Datetime                                                                     
2026-05-08 16:00:00+00:00  96.129997  96.180000  95.449997  95.809998   16343
2026-05-08 17:00:00+00:00  95.489998  96.160004  95.309998  96.129997   12887
2026-05-08 18:00:00+00:00  95.559998  95.809998  94.589996  95.489998   32283
2026-05-08 19:00:00+00:00  94.910004  95.650002  94.690002  95.559998    6059
2026-05-08 20:00:00+00:00  95.419998  95.419998  94.410004  94.919998    3009


### Calculate liquidity variables derived from the OHLCV entries

Four liquidity metrics are calculated from the raw OHLCV data:

- **log_volume:** Natural logarithm of hourly trading volume. Log transformation normalizes the right-skewed distribution of volume, making it more suitable for linear modeling.

- **price_range:** Difference between High and Low price within each hour. This serves as an intraday volatility proxy following the Parkinson (1980) extreme value estimator.

- **log_return:** Logarithmic return between consecutive hours, computed as log(Close_t / Close_{t-1}). Log returns are standard in financial analysis due to their additive properties and approximate normality.

- **amihud:** Ratio of absolute log return to trading volume, following Amihud (2002). This is the most widely cited illiquidity measure in market microstructure literature, capturing the price impact per unit of volume traded.

In [14]:
# Cell 4 — Compute WTI liquidity features
df_wti = data['wti'].copy()
df_wti['log_volume'] = np.log(df_wti['Volume'] + 1)
df_wti['price_range'] = df_wti['High'] - df_wti['Low']
df_wti['log_return'] = np.log(df_wti['Close'] / df_wti['Close'].shift(1))
df_wti['amihud'] = df_wti['log_return'].abs() / (df_wti['Volume'] + 1)

print("Liquidity features computed:")
print(df_wti[['log_volume', 'price_range', 'amihud']].describe().round(4))

Liquidity features computed:
Price  log_volume  price_range      amihud
count  11193.0000   11193.0000  11192.0000
mean       8.3224       0.4874      0.0001
std        2.1867       0.6173      0.0013
min        0.0000       0.0000      0.0000
25%        7.5137       0.1900      0.0000
50%        8.7276       0.3300      0.0000
75%        9.7732       0.5400      0.0000
max       13.3046      14.4000      0.1107


In [15]:
# Cell 5 — Build market_context and save to DB
df_dxy = data['dxy'][['Close']].rename(columns={'Close': 'dxy'})
df_vix = data['vix'][['Close']].rename(columns={'Close': 'vix'})

df_market = df_wti[['log_volume', 'price_range', 'log_return', 'amihud']].copy()
df_market = df_market.join(df_dxy, how='left')
df_market = df_market.join(df_vix, how='left')
df_market['vix'] = df_market['vix'].ffill(limit=16)

# Add empty Cushing column — will be filled in notebook 09
df_market['cushing_inventory'] = None
df_market = df_market.reset_index()
df_market['datetime_hour'] = df_market['datetime_hour'].astype(str)

df_market.to_sql('market_context', conn, if_exists='replace', index=False)

print(f"market_context saved: {len(df_market)} records")
print(f"DXY coverage: {df_market['dxy'].notna().sum()} / {len(df_market)}")
print(f"VIX coverage: {df_market['vix'].notna().sum()} / {len(df_market)}")

market_context saved: 11193 records
DXY coverage: 11186 / 11193
VIX coverage: 11187 / 11193


### Save data

In [16]:
# Cell 6 — Verify
sample = pd.read_sql("""
    SELECT datetime_hour, log_volume, price_range, dxy, vix
    FROM market_context
    WHERE dxy IS NOT NULL
    LIMIT 5
""", conn)
print(sample)

counts = pd.read_sql("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN dxy IS NOT NULL THEN 1 ELSE 0 END) as dxy_count,
        SUM(CASE WHEN vix IS NOT NULL THEN 1 ELSE 0 END) as vix_count
    FROM market_context
""", conn)
print(counts)

conn.close()
print("Done.")

               datetime_hour  log_volume  price_range         dxy    vix
0  2024-05-10 13:00:00+00:00    0.000000     0.459999  105.162003  12.89
1  2024-05-10 14:00:00+00:00   10.176221     0.519997  105.385002  12.84
2  2024-05-10 15:00:00+00:00   10.217678     0.639999  105.302002  12.78
3  2024-05-10 16:00:00+00:00   10.399798     0.510002  105.273003  12.73
4  2024-05-10 17:00:00+00:00    9.922702     0.250000  105.293999  12.71
   total  dxy_count  vix_count
0  11193      11186      11187
Done.
